# Examine sv-evidence-extraction output

Basic loading and sanity-check tools for the PE/SR/RD tables produced by
a `query`- or `build-tables`-mode run: discover what result sets are on
disk, load one into a dict of DataFrames, label each row with the
sample's family role (child/father/mother, via the pedigree), and get a
quick per-class, per-sample row-count summary before doing any real
analysis.

`local_evidence_dir` in `local_config.json` points at wherever you've
downloaded results to locally (gitignored, like the other workspace
paths -- see `local_config.example.json`).


In [2]:
# ==========================================
# IMPORTS
# ==========================================
import json
from pathlib import Path

import pandas as pd


In [3]:
# ==========================================
# LOCAL CONFIG
# ==========================================
# Assumes the notebook is run from its default location (notebooks/);
# adjust REPO_ROOT if you've moved it.
REPO_ROOT = Path.cwd().parent
LOCAL_CONFIG_PATH = REPO_ROOT / "local_config.json"

with open(LOCAL_CONFIG_PATH) as fh:
    local_config = json.load(fh)

DATA_DIR = Path(local_config["local_evidence_dir"])
DATA_DIR


PosixPath('/Users/murphyda/My Drive/SV_manual_plot_review/SV_evidence_extraction')

In [4]:
# ==========================================
# DISCOVER AVAILABLE RESULT SETS
# ==========================================
def list_available_prefixes(data_dir, evidence_class="pe", fmt="parquet"):
    """List the distinct output_prefix values with evidence tables present in a directory.

    Parameters
    ----------
    data_dir : str or pathlib.Path
        Directory containing sv-evidence-extraction output files, named
        "<prefix>.{pe,sr,rd}.{tsv,parquet}".
    evidence_class : {"pe", "sr", "rd"}, default "pe"
        Which evidence class's files to key off of -- any one class is
        enough to discover the prefix, since all three (when present)
        share the same prefix.
    fmt : {"tsv", "parquet"}, default "parquet"
        Which file extension to look for.

    Returns
    -------
    list of str
        Sorted, unique output_prefix values found.
    """
    data_dir = Path(data_dir)
    suffix = f".{evidence_class}.{fmt}"
    return sorted({p.name[: -len(suffix)] for p in data_dir.glob(f"*{suffix}")})


In [5]:
# ==========================================
# LOAD ONE RESULT SET
# ==========================================
EVIDENCE_CLASSES = ("pe", "sr", "rd")


def load_evidence_set(prefix, data_dir, fmt="parquet"):
    """Load the PE/SR/RD tables for one output_prefix into a dict of DataFrames.

    Parameters
    ----------
    prefix : str
        The output_prefix used when the evidence was extracted (matches
        the region_name from build_query_inputs.ipynb, e.g.
        "chr20_38104076_38108227_<child_id>").
    data_dir : str or pathlib.Path
        Directory containing the "<prefix>.{pe,sr,rd}.{tsv,parquet}" files.
    fmt : {"tsv", "parquet"}, default "parquet"
        Which file format to load -- parquet is smaller/faster locally;
        tsv is what the cluster-side WDL task also produces.

    Returns
    -------
    dict of str -> pandas.DataFrame
        Keys "pe", "sr", "rd". A missing file for a given class returns
        an empty DataFrame for that key rather than raising, so a
        partial result set still loads.
    """
    data_dir = Path(data_dir)
    reader = pd.read_parquet if fmt == "parquet" else (lambda p: pd.read_csv(p, sep="\t"))

    evidence = {}
    for evidence_class in EVIDENCE_CLASSES:
        path = data_dir / f"{prefix}.{evidence_class}.{fmt}"
        evidence[evidence_class] = reader(path) if path.exists() else pd.DataFrame()
    return evidence


In [6]:
# ==========================================
# SANITY-CHECK SUMMARY
# ==========================================
def summarize_evidence(evidence, prefix=None):
    """Print a quick per-class, per-sample row-count sanity check.

    Worth running before any real analysis: an empty table can mean
    "genuinely no evidence" but can just as easily mean the extraction
    silently failed to access the source file (this happened once during
    development -- see OPERATIONS.md). RD in particular should
    essentially never be empty for a real sample/region, since bincov
    matrices are dense, so an empty RD table is a stronger red flag than
    an empty PE/SR table.

    Parameters
    ----------
    evidence : dict of str -> pandas.DataFrame
        As returned by `load_evidence_set`, optionally already passed
        through `label_evidence_relationships` -- if a "relationship"
        column is present, counts are broken out by it.
    prefix : str, optional
        Label to print above the summary, for readability when checking
        several result sets in a row.

    Returns
    -------
    pandas.DataFrame
        One row per (evidence_class, sample_id[, relationship]), with a
        "rows" count column -- meant to be read at a glance, not joined
        onto anything.
    """
    if prefix:
        print(f"=== {prefix} ===")

    rows = []
    for evidence_class, df in evidence.items():
        if df.empty:
            print(f"  {evidence_class.upper():>3}: EMPTY (0 rows) -- verify this is really \"no evidence\", not an access failure")
            continue

        group_cols = ["relationship", "sample_id"] if "relationship" in df.columns else ["sample_id"]
        counts = df.groupby(group_cols).size()
        print(f"  {evidence_class.upper():>3}: {len(df)} rows across {df['sample_id'].nunique()} sample(s)")
        for key, count in counts.items():
            row = dict(zip(group_cols, key if isinstance(key, tuple) else (key,)))
            row["evidence_class"] = evidence_class
            row["rows"] = count
            rows.append(row)

    return pd.DataFrame(rows)


In [7]:
# ==========================================
# PEDIGREE LOOKUP
# ==========================================
# Shared with build_query_inputs.ipynb via pedigree_utils.py.
from pedigree_utils import load_pedigree, label_evidence_relationships

df_ped = load_pedigree(local_config["ped_file_uri"])
df_ped.head()


/opt/anaconda3/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


,FamID,IndividualID,FatherID,MotherID,Gender,Affected
0,11000,__ssc02217__0cb03e,0,0,2,1.0
1,11000,__ssc02219__4ee3b0,0,0,1,1.0
2,11000,__ssc02220__b84587,__ssc02219__4ee3b0,__ssc02217__0cb03e,2,1.0
3,11000,__ssc02254__21f98a,__ssc02219__4ee3b0,__ssc02217__0cb03e,1,2.0
4,11001,__ss0013024__920359,__ssc02184__a8235a,__ssc02181__ddf9cb,1,2.0


## Example: load and summarize one result set


In [8]:
# ==========================================
# EXAMPLE: LOAD, LABEL, AND SUMMARIZE ONE RESULT SET
# ==========================================
available = list_available_prefixes(DATA_DIR)
print(f"{len(available)} result set(s) found in {DATA_DIR}:")
for p in available:
    print(f"  {p}")

prefix = available[-1]
evidence = load_evidence_set(prefix, DATA_DIR)
# evidence = label_evidence_relationships(evidence, df_ped)
summary = summarize_evidence(evidence, prefix=prefix)
summary


2 result set(s) found in /Users/murphyda/My Drive/SV_manual_plot_review/SV_evidence_extraction:
  bulk_extraction_20260915
  chr20_38104076_38108227___2_1738_003_recal__661056
=== chr20_38104076_38108227___2_1738_003_recal__661056 ===
   PE: 30 rows across 3 sample(s)
   SR: 469 rows across 3 sample(s)
   RD: 378 rows across 3 sample(s)


,relationship,sample_id,evidence_class,rows
0,child,__2_1738_003_recal__661056,pe,14
1,father,__2_1738_002_recal__b3dabd,pe,2
2,mother,__2_1738_001_recal__07e559,pe,14
3,child,__2_1738_003_recal__661056,sr,242
4,father,__2_1738_002_recal__b3dabd,sr,112
5,mother,__2_1738_001_recal__07e559,sr,115
6,child,__2_1738_003_recal__661056,rd,126
7,father,__2_1738_002_recal__b3dabd,rd,126
8,mother,__2_1738_001_recal__07e559,rd,126


In [9]:
df = evidence['pe']
df = df[df.chrom1==df.chrom2]
df['pe_delta_abs'] = (df.pos1 - df.pos2).abs()
df = df[df.pe_delta_abs<1e4]
f_save = '/Users/murphyda/My Drive/SV_manual_plot_review/SV_evidence_extraction/chr20_38104076_38108227___2_1738_003_recal__661056.cleanPE.tsv'
df.to_csv(f_save, sep='\t', index=False)

/var/folders/fn/wx64g8qj6vz18p5fc0zspsz00000gq/T/ipykernel_84713/2753769896.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pe_delta_abs'] = (df.pos1 - df.pos2).abs()


In [10]:
df = evidence['sr']
df.sort_values(by=['count', 'pos'], ascending=[False, False])

,name,chrom,pos,orientation,count,sample_id,relationship
31,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38101054,right,15,__2_1738_001_recal__07e559,mother
30,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38101054,right,13,__2_1738_003_recal__661056,child
134,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38103383,left,10,__2_1738_002_recal__b3dabd,father
115,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38103233,right,10,__2_1738_002_recal__b3dabd,father
211,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38104209,left,9,__2_1738_001_recal__07e559,mother
...,...,...,...,...,...,...,...
4,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38100352,right,1,__2_1738_003_recal__661056,child
3,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38100307,right,1,__2_1738_001_recal__07e559,mother
2,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38100237,left,1,__2_1738_003_recal__661056,child
1,chr20_38104076_38108227___2_1738_003_recal__66...,chr20,38100196,left,1,__2_1738_003_recal__661056,child


In [11]:
# df = df[df.chrom1==df.chrom2]
# df['pe_delta_abs'] = (df.pos1 - df.pos2).abs()
# df = df[df.pe_delta_abs<1e4]
f_save = '/Users/murphyda/My Drive/SV_manual_plot_review/SV_evidence_extraction/chr20_38104076_38108227___2_1738_003_recal__661056.cleanSR.tsv'
df.to_csv(f_save, sep='\t', index=False)

In [12]:
df_rd = pd.read_parquet('/Users/murphyda/My Drive/SV_manual_plot_review/SV_evidence_extraction/bulk_extraction_20260915.rd.parquet')

In [13]:
df_rd

,name,chrom,start,end,sample_id,relationship,read_depth,median_cov
0,DEL_chr1_260___asd_1841__fcabd1,chr1,10000,10100,__asd_1841__fcabd1,child,561.0,20
1,DEL_chr1_260___asd_1841__fcabd1,chr1,10100,10200,__asd_1841__fcabd1,child,250.0,20
2,DEL_chr1_260___asd_1841__fcabd1,chr1,10200,10300,__asd_1841__fcabd1,child,289.0,20
3,DEL_chr1_260___asd_1841__fcabd1,chr1,10300,10400,__asd_1841__fcabd1,child,403.0,20
4,DEL_chr1_260___asd_1841__fcabd1,chr1,10400,10500,__asd_1841__fcabd1,child,63.0,20
...,...,...,...,...,...,...,...,...
17378107,DUP_chr14_825___sp0194908__27e6e9,chr14,31293313,31293413,__sp0194906__1987c3,mother,21.0,20
17378108,DUP_chr14_825___sp0194908__27e6e9,chr14,31293413,31293513,__sp0194906__1987c3,mother,19.0,20
17378109,DUP_chr14_825___sp0194908__27e6e9,chr14,31293513,31293613,__sp0194906__1987c3,mother,17.0,20
17378110,DUP_chr14_825___sp0194908__27e6e9,chr14,31293613,31293713,__sp0194906__1987c3,mother,33.0,20
